# Test post-génération du modèle entraîné sur la dataset "life_style_data"

## 1. Importation des librairies essentielles

In [1]:
from pathlib import Path
import joblib, json, yaml

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline

## 2. Exemple d’entrée simulée depuis l’interface utilisateur

Cette cellule permet de définir un dictionnaire Python (`ui_input`) reproduisant la structure exacte des données envoyées par l’interface Gradio. 

Chaque clé correspond au **nom d’une feature d’entrée** utilisée par le modèle.  

Nous n’activons pour l’instant que les variables *Age* et *Weight (kg)* — les autres seront intégrées dans les prochaines versions du pipeline (`Gender`, `Experience_Level`, etc.).


In [2]:
# === Simulation d'une entrée utilisateur (depuis l'UI) ===
# Le notebook s’exécute depuis son répertoire → on peut repartir du cwd
current_dir = Path.cwd()
project_root = current_dir.parents[3]  # remonte jusqu’à "train.me"

# Valeurs par défaut UI (non normalisées)
config_path = Path(project_root / "src/config/ui_defaults.yaml")
with open(config_path, "r", encoding="utf-8") as f:
    UI_DEFAULTS = yaml.safe_load(f)["UI_DEFAULTS"]

print("Exemple d'entrée utilisateur simulée :")
for k, v in UI_DEFAULTS.items():
    print(f" - {k}: {v}")

Exemple d'entrée utilisateur simulée :
 - Age: 55.92
 - Gender: Female
 - Weight (kg): 84.07
 - Height (m): 1.63
 - Workout_Type: Yoga
 - Workout_Frequency (days/week): 3.97


## 3. Définition des chemins du modèle et des fichiers associés

Cette cellule identifie dynamiquement la racine du projet `train.me` à partir du répertoire courant,  
puis construit les chemins complets vers :
- le modèle entraîné (`model.joblib`)  
- le scaler des features (`feature_scaler.joblib`)  
- le scaler de la variable cible (`target_scaler.joblib`)

Cela garantit que le notebook reste portable, même si le dossier est déplacé.


In [3]:
# === Localisation dynamique des fichiers du modèle ===
# On part du dossier actuel (celui du notebook)
current_dir = Path(__file__).resolve() if "__file__" in globals() else Path.cwd()

# Remonte jusqu’à la racine du projet "train.me"
project_root = current_dir.parents[3]
print(f"Racine du projet détectée : {project_root}")

# Définition des chemins vers le modèle et les objets de scaling
model_dir = project_root / "src" / "models" / "v1" / "life_style_data"

# Fichiers du pipeline ML
model_fp      = model_dir / "model.joblib"           # Modèle entraîné complet (pipeline)
fx_scaler_fp  = model_dir / "feature_scaler.joblib"  # Scaler utilisé pour normaliser les features
y_scaler_fp   = model_dir / "target_scaler.joblib"   # Scaler utilisé pour rescaler la target
gender_enc_fp  = model_dir / "encoder.joblib"

# Vérification rapide
print("Dossiers et fichiers cibles :")
print(f" - Modèle entraîné        : {model_fp}")
print(f" - Feature scaler          : {fx_scaler_fp}")
print(f" - Target scaler           : {y_scaler_fp}")
print(f" - Gender encoder         : {gender_enc_fp}")

Racine du projet détectée : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me
Dossiers et fichiers cibles :
 - Modèle entraîné        : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\models\v1\life_style_data\model.joblib
 - Feature scaler          : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\models\v1\life_style_data\feature_scaler.joblib
 - Target scaler           : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\models\v1\life_style_data\target_scaler.joblib
 - Gender encoder         : c:\Users\fback\Desktop\Projets\Dev\GitHub\train.me\src\models\v1\life_style_data\encoder.joblib


## 4. Chargement du modèle entraîné

Cette cellule charge le modèle sauvegardé lors de la phase d’entraînement.  

Le fichier `model.joblib` contient un pipeline complet (prétraitement, transformation, modèle).  

On récupère ensuite la liste des **features attendues** par ce modèle — utile pour vérifier la correspondance avec les données d’entrée simulées.


In [4]:
# Chargement du modèle
model = joblib.load(model_fp)

# Récupération robuste des features attendues
expected = None
if hasattr(model, "feature_names_in_"):
    expected = list(model.feature_names_in_)
else:
    # fallback via feature_schema.json
    feature_schema_fp = model_fp.parent / "feature_schema.json"
    if feature_schema_fp.exists():
        with open(feature_schema_fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        expected = data.get("feature_names_in_")
    if not expected:
        raise AttributeError(
            "Impossible de déterminer les colonnes attendues : ni "
            "`model.feature_names_in_` ni `feature_schema.json` trouvés."
        )

print(f"Modèle chargé : {model_fp.name}")
print(f"→ Features attendues ({len(expected)}) : {expected}")

Modèle chargé : model.joblib
→ Features attendues (8) : ['Age', 'Weight (kg)', 'Height (m)', 'Gender_1.0', 'Workout_Frequency (days/week)', 'Workout_Type_HIIT', 'Workout_Type_Strength', 'Workout_Type_Yoga']


### 4.1. Identification du modèle utilisé

Cette cellule permet d’afficher le type de modèle réellement chargé dans le pipeline.  

Selon la configuration du `model.joblib`, il peut s’agir d’une **régression linéaire**, d’un **Random Forest**, ou d’un **Gradient Boosting**.  

La fonction `model_friendly()` renvoie un nom lisible pour faciliter l’interprétation du rapport ou du log.


In [5]:
def model_friendly(estimator):
    """
    Retourne un nom lisible du modèle entraîné.
    
    Paramètres
    ----------
    estimator : objet scikit-learn
        Modèle ou pipeline entraîné.

    Retour
    ------
    str : nom du modèle en format humain
    """
    if isinstance(estimator, RandomForestRegressor):
        return "Random Forest"
    if isinstance(estimator, BaggingRegressor):
        return "Bagging Regressor"
    if isinstance(estimator, GradientBoostingRegressor):
        return "Gradient Boosting"
    if isinstance(estimator, LinearRegression):
        return "Régression Linéaire"
    return estimator.__class__.__name__

# === Affichage du type de modèle ===
print(f"Type de modèle : {model_friendly(model)}")

Type de modèle : Random Forest


## 5. Validation des entrées et prédiction

Cette cellule :
1. Vérifie que les variables d’entrée issues de `ui_input` correspondent bien aux colonnes attendues par le modèle (`expected`).  
2. Construit un `DataFrame` Pandas dans le bon ordre de colonnes et au bon format numérique.  
3. Réalise la prédiction à l’aide du modèle chargé et affiche la valeur estimée de **Experience_Level**.  

Cette étape simule exactement ce qui se passera lors de l’appel depuis l’interface Gradio.


In [6]:
fx_scaler = None
y_scaler = None
encoder: OrdinalEncoder | None = None

# Chargement "best effort"
if Path(fx_scaler_fp).exists():
    fx_scaler = joblib.load(fx_scaler_fp)
    print(f"Feature scaler chargé : {Path(fx_scaler_fp).name}")
else:
    print("ℹAucun feature scaler trouvé (on supposera un Pipeline intégrant le scaler).")

if Path(y_scaler_fp).exists():
    y_scaler = joblib.load(y_scaler_fp)
    print(f"Target scaler chargé : {Path(y_scaler_fp).name}")
else:
    print("ℹAucun target scaler trouvé (cible non normalisée).")

if gender_enc_fp.exists():
    encoder = joblib.load(gender_enc_fp)
    print(f"Gender encoder chargé : {gender_enc_fp.name}")
else:
    print("ℹAucun encoder.joblib trouvé — Gender_1.0 ne pourra pas être reconstruit.")

Feature scaler chargé : feature_scaler.joblib
Target scaler chargé : target_scaler.joblib
Gender encoder chargé : encoder.joblib


## 6. Construction des features brutes (Age, Weight, Gender_1.0, Experience_Level)

In [7]:
np.set_printoptions(precision=6, suppress=True)

def _pipeline_has_scaler(p):
    return isinstance(p, Pipeline) and any(
        isinstance(step, (StandardScaler, MinMaxScaler, RobustScaler))
        for _, step in p.named_steps.items()
    )

In [8]:
def ui_to_internal_row(ui_dict: dict, expected_cols: list[str]) -> pd.DataFrame:
    """
    Transforme un dictionnaire d'entrée UI
    en DataFrame aligné sur les colonnes internes attendues par le modèle.

    Gestion spéciale :
    - Gender_1.0 : reconstruit via encoder + dummy binaire
    - Workout_Type_* : colonnes One-Hot reconstruites à partir de 'Workout_Type'
    - Body Part_* : colonnes One-Hot reconstruites à partir de 'Body Part'
    - Equipment Needed_* : colonnes One-Hot reconstruites à partir de 'Equipment Needed'
    - Difficulty Level : mapping texte -> entier (0,1,2)
    - le reste : cast en float depuis ui_dict
    """
    row: dict[str, float] = {}

    for col in expected_cols:
        # --- Cas spécial : Gender_1.0 ---
        if col == "Gender_1.0":
            if encoder is None:
                raise RuntimeError(
                    "Le modèle attend la colonne 'Gender_1.0' mais aucun encoder.joblib "
                    "n'a été trouvé."
                )
            if "Gender" not in ui_dict:
                raise KeyError("Clé 'Gender' manquante dans ui_input.")

            g_str = ui_dict["Gender"]
            g_df = pd.DataFrame([[g_str]], columns=["Gender"])
            g_encoded = float(encoder.transform(g_df)[0, 0])
            row["Gender_1.0"] = 1.0 if g_encoded == 1.0 else 0.0

        # --- Cas spécial : colonnes One-Hot Workout_Type_* ---
        elif col.startswith("Workout_Type_"):
            if "Workout_Type" not in ui_dict:
                raise KeyError("Clé 'Workout_Type' manquante dans ui_input.")

            w_str = str(ui_dict["Workout_Type"]).strip()
            expected_cat = col.split("Workout_Type_", 1)[1]
            row[col] = 1.0 if w_str == expected_cat else 0.0

        # # --- Cas spécial : colonnes One-Hot Body Part_* ---
        # elif col.startswith("Body Part_"):
        #     if "Body Part" not in ui_dict:
        #         raise KeyError("Clé 'Body Part' manquante dans ui_input.")

        #     b_str = str(ui_dict["Body Part"]).strip()
        #     expected_cat = col.split("Body Part_", 1)[1]
        #     row[col] = 1.0 if b_str == expected_cat else 0.0


        # --- Cas général : features numériques (Age, Weight, BPM, etc.) ---
        else:
            if col not in ui_dict:
                raise KeyError(f"Clé '{col}' manquante dans ui_input.")
            row[col] = float(ui_dict[col])

    return pd.DataFrame([row], columns=expected_cols)

In [9]:
# --- Entrée UI → DataFrame brut (Age, Weight, Gender_1.0) ---
print(f"Entrées UI : {UI_DEFAULTS}")
X_one_raw = ui_to_internal_row(UI_DEFAULTS, expected)

# --- Scaling des features ---
uses_internal_scaling = _pipeline_has_scaler(model)
if uses_internal_scaling:
    X_one = X_one_raw.copy()
    print("Pipeline: scaler interne → pas de double-scaling.")
else:
    if fx_scaler is None:
        raise RuntimeError(
            "Pas de scaler interne dans le modèle et aucun feature_scaler.joblib trouvé."
        )
    X_one = pd.DataFrame(fx_scaler.transform(X_one_raw), columns=expected)
    print("Scaling appliqué via feature_scaler.joblib.")

# --- Infos scaler (diagnostic léger) ---
print("feature_scaler.mean_ :", getattr(fx_scaler, "mean_", None))
print("feature_scaler.scale_:", getattr(fx_scaler, "scale_", None))

# Vérif cohérence (Z-score manuel vs transform)
z_manual = (X_one_raw - fx_scaler.mean_) / fx_scaler.scale_
z_auto   = pd.DataFrame(fx_scaler.transform(X_one_raw), columns=expected)
# print("Z-manual:\n", z_manual)
# print("Z-auto  :\n", z_auto)

# Prédiction modèle → extrait proprement une valeur scalaire
y_std_one = model.predict(X_one)[0]   # extrait la valeur
y_std_one = float(y_std_one)          # conversion propre sans warning

if y_scaler is not None:
    y_kcal_one = y_scaler.inverse_transform([[y_std_one]])[0, 0]
    print(f"UI → y_std: {y_std_one:.6f} | (entre 1 et 3.05)): {y_kcal_one:.2f}")
else:
    print(f"UI → y_std: {y_std_one:.6f} (pas de target_scaler)")

Entrées UI : {'Age': 55.92, 'Gender': 'Female', 'Weight (kg)': 84.07, 'Height (m)': 1.63, 'Workout_Type': 'Yoga', 'Workout_Frequency (days/week)': 3.97}
Scaling appliqué via feature_scaler.joblib.
feature_scaler.mean_ : [38.88854989 73.82786164  1.72307033  0.49807033  3.318755    0.24878502
  0.25743282  0.24978559]
feature_scaler.scale_: [12.11858176 21.10416305  0.12728256  0.49999628  0.9122416   0.43230896
  0.43721981  0.43288884]
UI → y_std: 0.262089 | (entre 1 et 3.05)): 2.00
